<a href="https://colab.research.google.com/github/KP-365/Skinrash-detection/blob/main/BugBitesAug.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from datasets import load_from_disk, concatenate_datasets, DatasetDict
import numpy as np
from collections import Counter
from sklearn.model_selection import train_test_split
from collections import Counter


In [7]:
"""
True 70/15/15 split, computed across the FULL combined pool
(augmented train+val + clean train+val+test), so percentages
are accurate against the whole dataset -- not just a subset.

Leakage safeguard: clean test images are pulled out FIRST and
never touched by augmentation-derived duplicates.
"""

aug = load_dataset("eceunal/bug-bite-images-aug_v3")
clean = load_dataset("eceunal/bug-bite-images-hf")

labels = aug["train"].features["label"].names

# Full pool = augmented (train+val) + clean (train+val)
# Clean's ORIGINAL test split is reserved separately and NOT pooled,
# to guarantee test images are genuinely unseen originals.
pool = concatenate_datasets([
    aug["train"], aug["validation"],
    clean["train"], clean["validation"]
])

reserved_test = clean["test"]  # 53 clean images, held out entirely

total_target = len(pool) + len(reserved_test)
print(f"Full combined size: {total_target}")
print(f"Reserved clean test: {len(reserved_test)} ({100*len(reserved_test)/total_target:.1f}% of total)")

# Since reserved_test is fixed at 53, work out what remains for train/val
# to hit true 70/15/15 as closely as possible.
target_test_frac = 0.15
target_test_n = int(round(total_target * target_test_frac))
extra_test_needed = target_test_n - len(reserved_test)

print(f"Target test size for true 15%: {target_test_n}")
print(f"Additional test images needed from pool: {extra_test_needed}")

indices = np.arange(len(pool))
label_col = pool["label"]

if extra_test_needed > 0:
    # Carve out additional test images from the pool, stratified
    remaining_idx, extra_test_idx = train_test_split(
        indices, test_size=extra_test_needed, stratify=label_col, random_state=1337
    )
    extra_test = pool.select(extra_test_idx)
    full_test = concatenate_datasets([reserved_test, extra_test])
    pool = pool.select(remaining_idx)
    indices = np.arange(len(pool))
    label_col = pool["label"]
else:
    full_test = reserved_test

# Now split remaining pool into 70/15 (train/val) relative to ORIGINAL total
remaining_target_train_frac = 0.70 / (0.70 + 0.15)  # train's share of what's left
train_idx, val_idx = train_test_split(
    indices, train_size=remaining_target_train_frac,
    stratify=label_col, random_state=1337
)

final = DatasetDict({
    "train": pool.select(train_idx),
    "validation": pool.select(val_idx),
    "test": full_test
})

total = sum(len(final[s]) for s in final)
print(f"\nFinal split (target 70/15/15, total={total}):")
for split in final:
    pct = 100 * len(final[split]) / total
    print(f"  {split}: {len(final[split])} ({pct:.1f}%)")

print("\nPer-class breakdown:")
for split in final:
    counts = Counter(final[split]["label"])
    s_total = len(final[split])
    print(f"\n{split} ({s_total} total):")
    for i, name in enumerate(labels):
        c = counts.get(i, 0)
        pct = 100 * c / s_total if s_total else 0
        print(f"  {name}: {c} ({pct:.1f}%)")

final.save_to_disk("bug_bite_final_split_70_15_15")
print("\nSaved to bug_bite_final_split_70_15_15")

Full combined size: 9331
Reserved clean test: 53 (0.6% of total)
Target test size for true 15%: 1400
Additional test images needed from pool: 1347

Final split (target 70/15/15, total=9331):
  train: 6531 (70.0%)
  validation: 1400 (15.0%)
  test: 1400 (15.0%)

Per-class breakdown:

train (6531 total):
  ants: 899 (13.8%)
  bed_bugs: 913 (14.0%)
  chiggers: 832 (12.7%)
  fleas: 783 (12.0%)
  mosquitos: 648 (9.9%)
  no_bites: 788 (12.1%)
  spiders: 822 (12.6%)
  ticks: 846 (13.0%)

validation (1400 total):
  ants: 193 (13.8%)
  bed_bugs: 196 (14.0%)
  chiggers: 178 (12.7%)
  fleas: 168 (12.0%)
  mosquitos: 139 (9.9%)
  no_bites: 169 (12.1%)
  spiders: 176 (12.6%)
  ticks: 181 (12.9%)

test (1400 total):
  ants: 195 (13.9%)
  bed_bugs: 195 (13.9%)
  chiggers: 179 (12.8%)
  fleas: 163 (11.6%)
  mosquitos: 138 (9.9%)
  no_bites: 171 (12.2%)
  spiders: 178 (12.7%)
  ticks: 181 (12.9%)


Saving the dataset (0/1 shards):   0%|          | 0/6531 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1400 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1400 [00:00<?, ? examples/s]


Saved to bug_bite_final_split_70_15_15


In [12]:
SEED = 1337
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
np.random.seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_CLASSES = 8
BATCH_SIZE = 32
IMG_SIZE = 224
EPOCHS_HEAD = 50
EPOCHS_FINETUNE = 20
LR_HEAD = 1e-3
LR_FINETUNE = 1e-5
DROPOUT_P = 0.3
PATIENCE = 5

class EarlyStopping:
    """Tracks validation LOSS to decide when to stop training."""
    def __init__(self, patience=5, min_delta=0.0):
        self.patience = patience
        self.min_delta = min_delta
        self.best_loss = float('inf')
        self.counter = 0
        self.should_stop = False

    def __call__(self, val_loss):
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.should_stop = True

ds = load_from_disk("bug_bite_final_split_70_15_15")
labels = ds["train"].features["label"].names
print("Classes:", labels)

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.8, 1.0)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

class BugBiteDataset(Dataset):
    def __init__(self, hf_split, transform):
        self.data = hf_split
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        ex = self.data[idx]
        img = ex["image"].convert("RGB")
        img = self.transform(img)
        return img, ex["label"]

train_ds = BugBiteDataset(ds["train"], train_transform)
val_ds = BugBiteDataset(ds["validation"], eval_transform)
test_ds = BugBiteDataset(ds["test"], eval_transform)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

def build_model():
    model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
    for param in model.features.parameters():
        param.requires_grad = False
    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=DROPOUT_P),
        nn.Linear(in_features, NUM_CLASSES)
    )
    return model.to(DEVICE)

model = build_model()

def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for imgs, targets in loader:
        imgs, targets = imgs.to(DEVICE), targets.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * imgs.size(0)
        correct += (outputs.argmax(1) == targets).sum().item()
        total += imgs.size(0)
    return total_loss / total, correct / total

def eval_epoch(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    with torch.no_grad():
        for imgs, targets in loader:
            imgs, targets = imgs.to(DEVICE), targets.to(DEVICE)
            outputs = model(imgs)
            loss = criterion(outputs, targets)
            total_loss += loss.item() * imgs.size(0)
            correct += (outputs.argmax(1) == targets).sum().item()
            total += imgs.size(0)
    return total_loss / total, correct / total

criterion = nn.CrossEntropyLoss()

# ============================================================
# Phase 1: head-only training (frozen backbone)
# ============================================================
optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=LR_HEAD)
early_stopper = EarlyStopping(patience=PATIENCE)

print("\n=== Phase 1: Head-only training (frozen backbone) ===")
best_val_acc = 0
best_epoch_phase1 = 0
for epoch in range(EPOCHS_HEAD):
    train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion)
    val_loss, val_acc = eval_epoch(model, val_loader, criterion)
    print(f"Epoch {epoch+1}/{EPOCHS_HEAD} | train_loss={train_loss:.4f} train_acc={train_acc:.4f} | val_loss={val_loss:.4f} val_acc={val_acc:.4f}")

    # Save checkpoint ONLY when validation accuracy improves --
    # this file always holds the best-accuracy epoch's weights, not the latest epoch.
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_epoch_phase1 = epoch + 1
        torch.save(model.state_dict(), "best_model_phase1.pt")
        print(f"  -> New best (epoch {epoch+1}), checkpoint saved.")

    early_stopper(val_loss)
    if early_stopper.should_stop:
        print(f"Early stopping triggered at epoch {epoch+1} (phase 1)")
        break

print(f"\nPhase 1 complete. Best val_acc={best_val_acc:.4f} at epoch {best_epoch_phase1}")

# ============================================================
# Phase 2: fine-tune last block
# ============================================================
print("\n=== Phase 2: Fine-tuning last block ===")
model.load_state_dict(torch.load("best_model_phase1.pt"))  # start from phase 1's BEST weights
for param in model.features[-1].parameters():
    param.requires_grad = True

optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=LR_FINETUNE)
early_stopper = EarlyStopping(patience=PATIENCE)  # reset for phase 2

best_val_acc = 0
best_epoch_phase2 = 0
for epoch in range(EPOCHS_FINETUNE):
    train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion)
    val_loss, val_acc = eval_epoch(model, val_loader, criterion)
    print(f"Epoch {epoch+1}/{EPOCHS_FINETUNE} | train_loss={train_loss:.4f} train_acc={train_acc:.4f} | val_loss={val_loss:.4f} val_acc={val_acc:.4f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_epoch_phase2 = epoch + 1
        torch.save(model.state_dict(), "best_model_final.pt")
        print(f"  -> New best (epoch {epoch+1}), checkpoint saved.")

    early_stopper(val_loss)
    if early_stopper.should_stop:
        print(f"Early stopping triggered at epoch {epoch+1} (phase 2)")
        break

print(f"\nPhase 2 complete. Best val_acc={best_val_acc:.4f} at epoch {best_epoch_phase2}")
print(f"Final saved checkpoint: best_model_final.pt (from phase 2, epoch {best_epoch_phase2})")

# ============================================================
# Final test evaluation -- loads the BEST checkpoint, not the last epoch
# ============================================================
model.load_state_dict(torch.load("best_model_final.pt"))
test_loss, test_acc = eval_epoch(model, test_loader, criterion)
print(f"\nFinal TEST accuracy: {test_acc:.4f} | test_loss: {test_loss:.4f}")

Classes: ['ants', 'bed_bugs', 'chiggers', 'fleas', 'mosquitos', 'no_bites', 'spiders', 'ticks']

=== Phase 1: Head-only training (frozen backbone) ===
Epoch 1/50 | train_loss=1.5768 train_acc=0.4479 | val_loss=1.3511 val_acc=0.5414
  -> New best (epoch 1), checkpoint saved.
Epoch 2/50 | train_loss=1.2750 train_acc=0.5494 | val_loss=1.2203 val_acc=0.5779
  -> New best (epoch 2), checkpoint saved.
Epoch 3/50 | train_loss=1.1912 train_acc=0.5785 | val_loss=1.1181 val_acc=0.6321
  -> New best (epoch 3), checkpoint saved.
Epoch 4/50 | train_loss=1.1380 train_acc=0.5939 | val_loss=1.0781 val_acc=0.6300
Epoch 5/50 | train_loss=1.1172 train_acc=0.6109 | val_loss=1.0611 val_acc=0.6271
Epoch 6/50 | train_loss=1.0928 train_acc=0.6143 | val_loss=1.0287 val_acc=0.6507
  -> New best (epoch 6), checkpoint saved.
Epoch 7/50 | train_loss=1.0711 train_acc=0.6181 | val_loss=1.0195 val_acc=0.6571
  -> New best (epoch 7), checkpoint saved.
Epoch 8/50 | train_loss=1.0652 train_acc=0.6189 | val_loss=1.0163 v